Chapter 6 covers Anomaly Detection.

In quantitative risk management, financial engineering, and system monitoring, anomaly detection in SQL focuses on identifying statistical outliers, sudden regime shifts, unexpected spikes, or missing data points directly at the database tier before downstream models or automated pipelines consume the data.

The 4 Core Takeaways of Chapter 6

1. Statistical Outlier Detection (Z-Score & Interquartile Range)To detect whether a data point is anomalous relative to its historical or group distribution, SQL utilizes window aggregations to compute parametric or non-parametric dispersion bounds.

Parametric ($Z$-score): Compute population mean $\mu$ and standard deviation $\sigma$ across a window partition. Points where $\vert{}Z\vert{} > 3$ are flagged as statistical anomalies.

$$Z = \frac{x - \mu}{\sigma}$$

Non-Parametric (IQR): Use percentile functions (PERCENTILE_CONT(0.25) and PERCENTILE_CONT(0.75)) to flag points falling outside $[Q_1 - 1.5 \times \text{IQR}, Q_3 + 1.5 \times \text{IQR}]$.

2. Relative Deviation & Rolling Window Anomaly Thresholds

Fixed global thresholds fail when data exhibits structural trends or seasonality. SQL addresses this by computing moving statistics (rolling mean and rolling standard deviation) over rolling temporal windows.

The Pattern: Compute rolling standard deviation using STDDEV_SAMP(val) OVER (ORDER BY time_col ROWS BETWEEN N PRECEDING AND 1 PRECEDING) to avoid look-ahead bias, flagging current observations that deviate significantly from recent baseline behavior.

3. Step-Change & Structural Break DetectionAnomalies are not always extreme single-point spikes—sometimes they manifest as sudden step-changes in baseline level or volatility.

Deltas vs. Prior Periods: Combining LAG() with percentage change metrics allows you to flag sudden point-to-point structural shifts (e.g., a single-day transaction volume jump of > 500%).

4. Detecting Gaps in Expected Sequences (Missing Data Anomalies)

In time-series logs (sensor streams, daily price feeds, transaction logs), an anomaly can be the total absence of expected data.

The Pattern: Compare actual record counts against a generated continuous sequence (GENERATE_SERIES) using a LEFT JOIN where the event side yields NULL.


In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(database=":memory:")

# 1. Generate Synthetic Daily Transaction Volume with Injected Outliers
conn.execute("""
CREATE TABLE daily_metrics AS
SELECT
    d::DATE AS metric_date,
    -- Normal distribution around $100k + random noise + deliberate injected anomalies
    CASE
        WHEN d = DATE '2026-03-15' THEN 450000.00 -- Outlier Spike
        WHEN d = DATE '2026-04-10' THEN 5000.00   -- Outlier Drop
        ELSE ROUND(100000 + (random() * 20000 - 10000), 2)
    END AS transaction_volume
FROM range(DATE '2026-01-01', DATE '2026-05-01', INTERVAL 1 DAY) t(d);
""")

# 2. Chapter 6 Anomaly Detection Pipeline (Z-Score & Rolling Volatility Outliers)
anomalies_df = conn.execute("""
WITH rolling_stats AS (
    SELECT
        metric_date,
        transaction_volume,

        # 1. Compute Historical Population Baseline Stats
        AVG(transaction_volume) OVER () AS global_mean,
        STDDEV_SAMP(transaction_volume) OVER () AS global_stddev,

        # 2. Compute 14-Day Rolling Baseline (Excluding Current Row to prevent bias)
        AVG(transaction_volume) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
        ) AS rolling_14d_mean,

        STDDEV_SAMP(transaction_volume) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 14 PRECEDING AND 1 PRECEDING
        ) AS rolling_14d_stddev
    FROM daily_metrics
),
z_scores AS (
    SELECT
        metric_date,
        transaction_volume,
        rolling_14d_mean,

        # Calculate Parametric Z-Score relative to 14-day rolling window
        ROUND((transaction_volume - rolling_14d_mean) / NULLIF(rolling_14d_stddev, 0), 2) AS z_score_14d
    FROM rolling_stats
)
SELECT
    metric_date,
    transaction_volume,
    rolling_14d_mean,
    z_score_14d,

    # Flag Anomalies where |Z| > 3.0
    CASE
        WHEN z_score_14d > 3.0 THEN 'ANOMALY: High Spike'
        WHEN z_score_14d < -3.0 THEN 'ANOMALY: Severe Drop'
        ELSE 'Normal'
    END AS anomaly_flag
FROM z_scores
WHERE ABS(z_score_14d) > 3.0 OR metric_date IN ('2026-03-15', '2026-04-10');
""").df()

anomalies_df

| Concept                     | Problem Solved                                           | Primary SQL Syntax / Technique                                    |
| :-------------------------- | :------------------------------------------------------- | :---------------------------------------------------------------- |
| **Parametric Outliers**     | Flagging global distribution anomalies                   | `(x - AVG(x) OVER()) / STDDEV(x) OVER()`                          |
| **Rolling Anomalies**       | Accounting for seasonality/trends before flagging spikes | Moving window frame (`ROWS BETWEEN N PRECEDING AND 1 PRECEDING`)  |
| **Non-Parametric Outliers** | Detecting extreme values in skewed distributions (IQR)   | `PERCENTILE_CONT(0.75) - PERCENTILE_CONT(0.25)`                   |
| **Missing Sequence Gaps**   | Detecting missing logs or feeds                          | `GENERATE_SERIES(...) LEFT JOIN logs ON ... WHERE log.id IS NULL` |
